# Entrenamiento por experimento: LeNet-5 y MobileNetV1

La investigación requiere **12 experimentos**, pero este notebook entrena **uno por ejecución**. Conservar la misma `SUITE` entre sesiones permite acumular resultados y reanudar desde checkpoints por época. Ejecutar todas las celdas entrena un solo trial y deja test pendiente.

Consulte [el protocolo y los diagramas](../docs/architectures.md) y [la guía](../docs/training.md). La adaptación de los códigos RBF está documentada para revisión con el profesor.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
import torch
import wandb
from IPython.display import display, Markdown, Image

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
            if (p / 'ai/src/train.py').is_file())
AI = ROOT / 'ai'
DATA_ENV = os.environ.get('SPEECH_COMMANDS_TRAINING_DATA_DIR')
if DATA_ENV:
    DATA_DIR = Path(DATA_ENV).resolve()
else:
    exports = sorted(p for p in (AI / 'data/processed/speech_commands').glob('*')
                     if p.is_dir() and (p / 'COMPLETED.json').is_file() and (p / 'preprocessing.json').is_file()
                     and len(json.loads((p / 'preprocessing.json').read_text(encoding='utf-8'))['classes']) == 11)
    if not exports:
        raise FileNotFoundError('Primero exporte las once clases con el notebook 01 o configure SPEECH_COMMANDS_TRAINING_DATA_DIR')
    DATA_DIR = exports[-1]
CONFIG = AI / 'configs/experiments.json'
PROJECT = os.environ.get('WANDB_PROJECT', 'onnxmobile-speech-commands')
ENTITY = os.environ.get('WANDB_ENTITY')
WANDB_MODE = os.environ.get('WANDB_MODE', 'online')  # online u offline; tracking obligatorio
SUITE = AI / 'runs/speech_commands_gpu_augmented_v1'  # Suite nueva para comparar train original y expandido
EXPERIMENT = None  # None = siguiente pendiente; o 'lenet5-base-adam_1e-3'
EPOCHS_PER_RUN = 5  # Por ejemplo 5 para pausar tras cinco épocas completas
RUN_TEST = False  # Habilitar cuando quiera evaluar los cuatro ganadores
DEVICE = 'cuda'
if (SUITE / 'suite.json').exists():
    saved = json.loads((SUITE / 'suite.json').read_text(encoding='utf-8'))
    DATA_DIR = Path(json.loads((SUITE / 'dataset.json').read_text(encoding='utf-8'))['root'])
    PROJECT, ENTITY, WANDB_MODE = saved['project'], saved['entity'], saved['wandb_mode']
print('Suite:', SUITE)
print('Experimento:', EXPERIMENT or 'siguiente pendiente')
print('Datos:', DATA_DIR)
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())
print('W&B:', PROJECT, '| entidad:', ENTITY, '| modo:', WANDB_MODE)

Suite: d:\Repos\OnnxMobile\ai\runs\speech_commands
Experimento: siguiente pendiente
Datos: d:\Repos\OnnxMobile\ai\data\processed\speech_commands\20261008T020826106877Z
PyTorch: 2.5.1+cpu | CUDA disponible: False
W&B: onnxmobile-speech-commands | entidad: None | modo: online


## Arquitecturas y decisiones

Las capas, conexiones, pérdida, optimizadores, aumento y decisiones para uso móvil se describen a continuación. Los diagramas Mermaid también se pueden ver en `ai/docs/architectures.md` desde el repositorio.

In [ ]:
display(Markdown((AI / 'docs/architectures.md').read_text(encoding='utf-8')))

## Preparar y revisar ambos datasets

Este paso guarda train original + dos copias SpecAugment por ejemplo en una carpeta separada (3× train), sin iniciar entrenamiento. Cada copia usa time masking y frequency masking. Validation/test se comparten sin cambios. Los tensores originales se conservan; `preview.png` muestra las máscaras. Para once clases se esperan 83.150 ejemplos base y 249.450 aumentados.

In [ ]:
AUGMENTED_DIR = DATA_DIR.with_name(DATA_DIR.name + '-augmented')
subprocess.run([sys.executable, str(AI / 'src/augment.py'), '--data', str(DATA_DIR),
                '--output', str(AUGMENTED_DIR), '--config', str(CONFIG)], cwd=ROOT, check=True)
augmentation = json.loads((AUGMENTED_DIR / 'augmentation.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'dataset': 'base', **augmentation['source_counts']},
                      {'dataset': 'augmented', **augmentation['counts']}]))
display(Image(filename=str(AUGMENTED_DIR / 'preview.png')))

## Revisar la matriz y autenticar

Se selecciona por macro-F1 de validación. Base usa el dataset original y aumentado usa los originales más dos copias guardadas con máscaras temporales y espectrales. `train_clean` mide solo los originales. Validation/test y las particiones por hablante son comunes a todos los trials.

El login de W&B no se guarda en el notebook. En modo offline se registran eventos para sincronizarlos posteriormente.

In [2]:
sys.path.insert(0, str(AI / 'src'))
from train import experiment_grid
config = (json.loads((SUITE / 'suite.json').read_text(encoding='utf-8'))['config']
          if (SUITE / 'suite.json').exists() else json.loads(CONFIG.read_text(encoding='utf-8')))
grid = experiment_grid(config)
display(pd.DataFrame(grid)[['experiment', 'architecture', 'variant', 'optimizer', 'lr', 'batch_size', 'weight_decay', 'seed', 'epochs']])
if WANDB_MODE == 'online':
    wandb.login()
elif WANDB_MODE != 'offline':
    raise ValueError('W&B debe estar online u offline')

,experiment,architecture,variant,optimizer,lr,batch_size,weight_decay,seed,epochs
0,lenet5-base-adam_1e-3,lenet5,base,adam,0.0010,128,0.0000,42,30
1,lenet5-base-adam_3e-4,lenet5,base,adam,0.0003,64,0.0001,42,30
2,lenet5-base-sgd_1e-2,lenet5,base,sgd,0.0100,128,0.0001,42,30
3,lenet5-augmented-adam_1e-3,lenet5,augmented,adam,0.0010,128,0.0000,42,30
4,lenet5-augmented-adam_3e-4,lenet5,augmented,adam,0.0003,64,0.0001,42,30
5,lenet5-augmented-sgd_1e-2,lenet5,augmented,sgd,0.0100,128,0.0001,42,30
6,mobilenet_v1-base-adam_1e-3,mobilenet_v1,base,adam,0.0010,128,0.0000,42,30
7,mobilenet_v1-base-adam_3e-4,mobilenet_v1,base,adam,0.0003,64,0.0001,42,30
8,mobilenet_v1-base-sgd_1e-2,mobilenet_v1,base,sgd,0.0100,128,0.0001,42,30
9,mobilenet_v1-augmented-adam_1e-3,mobilenet_v1,augmented,adam,0.0010,128,0.0000,42,30


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\danie\_netrc.
wandb: Currently logged in as: danielpulido (danielpulido-instituto-tecnol-gico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Entrenar o continuar UN experimento

Esta celda no usa `--all`. Con `EXPERIMENT=None`, reejecutarla continúa el primer trial pendiente o inicia el siguiente. Indicar un nombre permite elegir otro; los completos se omiten.

Cada época guarda `last.pt` atómicamente con modelo, optimizador, RNG, historial y paciencia. `best.pt` conserva el mejor por validación. Si se interrumpe dentro de una época, esa época se repite al continuar. `EPOCHS_PER_RUN` permite pausas controladas.

In [ ]:
def execute(command):
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                          env={**os.environ, 'PYTHONIOENCODING': 'utf-8'}) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait() != 0:
                raise RuntimeError('Falló el comando; el progreso guardado se conserva')
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print('Interrumpido: la próxima ejecución usa la última época completa guardada.')
            raise

command = [sys.executable, str(AI / 'src/train.py'), '--suite', str(SUITE), '--watch']
if not (SUITE / 'suite.json').exists():
    command += ['--data', str(DATA_DIR), '--config', str(CONFIG), '--project', PROJECT,
                '--wandb-mode', WANDB_MODE, '--device', DEVICE]
    if ENTITY:
        command += ['--entity', ENTITY]
if EXPERIMENT:
    command += ['--experiment', EXPERIMENT]
if EPOCHS_PER_RUN is not None:
    command += ['--epochs-per-run', str(EPOCHS_PER_RUN)]
execute(command)
execute([sys.executable, str(AI / 'src/train.py'), '--suite', str(SUITE), '--list'])

Suite: D:\Repos\OnnxMobile\ai\runs\speech_commands
Device: cpu
Complete: 0/12
This invocation: 1
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\danie\_netrc.
wandb: Currently logged in as: danielpulido (danielpulido-instituto-tecnol-gico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: setting up run 798k2kjj
wandb: Tracking run with wandb version 0.29.0
wandb: Run data is saved locally in D:\Repos\OnnxMobile\ai\runs\speech_commands\wandb\run-20261008_180029-798k2kjj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lenet5-base-adam_1e-3
wandb:  View project at https://wandb.ai/danielpulido-instituto-tecnol-gico-de-costa-rica/onnxmobile-speech-commands
wandb:  View run at https://wandb.ai/danielpulido-instituto-tecnol-gico-de-costa-rica/onnxmobile-speech-commands/runs/798k2kjj


## Resultados acumulados

La tabla incorpora todos los trials completos, aunque se ejecuten en días distintos. Las curvas también se guardan para trials pausados. Compare train_clean con validación para discutir ajuste. Los ganadores se fijan cuando se completan los doce experimentos.

In [ ]:
if (SUITE / 'training_results.csv').exists():
    display(pd.read_csv(SUITE / 'training_results.csv'))
else:
    print('Todavía no hay trials completos; los checkpoints por época se conservan.')
for item in grid:
    path = SUITE / item['experiment'] / 'learning_curves.png'
    if path.exists():
        print(item['experiment'])
        display(Image(filename=str(path)))
if (SUITE / 'selection.json').exists():
    selection = json.loads((SUITE / 'selection.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(selection['winners'].values()))
    print('Mejor global por validación:', selection['best_overall'])
    print('Mejor base por validación:', selection['best_base'])
else:
    print('Se fijarán los cuatro ganadores cuando se completen los doce experimentos.')

## Test, en otra sesión si se prefiere

La evaluación se habilita con `RUN_TEST=True` después de completar los doce trials. Por defecto esta celda solo muestra resultados existentes. Test no decide hiperparámetros ni checkpoints. La evaluación conserva los resultados de cada ganador si se interrumpe.

In [ ]:
if RUN_TEST and not (SUITE / 'selection.json').exists():
    raise RuntimeError('Primero complete los doce experimentos y la selección por validación')
if RUN_TEST and not (SUITE / 'test_results.json').exists():
    execute([sys.executable, str(AI / 'src/evaluate.py'), '--suite', str(SUITE)])
if (SUITE / 'test_results.json').exists():
    display(pd.read_csv(SUITE / 'test_results.csv'))
    display(Markdown((SUITE / 'report.md').read_text(encoding='utf-8')))
else:
    print('Test pendiente; RUN_TEST=False conserva esta fase para otra sesión.')
print('Artefactos:', SUITE)
print('W&B: agrupar los runs por', SUITE.name)